In [6]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [7]:
df = pd.read_csv('movie-data/cleaned_analysis_data.csv')

print(df.columns)

# Removing columns which aren't needed (or directly related to un-profitability).
df.drop(columns=["id", "title", "release_date", "profit", "revenue", "budget"], inplace=True)

# Sorting the data by release date. (lower indexes are earlier dates)
# df.sort_values(by="release_year", inplace=True)

print(df.columns)
# print(df.isna().sum())

Index(['id', 'title', 'vote_average', 'vote_count', 'release_date', 'revenue',
       'runtime', 'budget', 'popularity', 'actor_avg', 'actor_med',
       'actor_dev', 'production_avg', 'production_med', 'production_dev',
       'release_year', 'release_month', 'original_title_matches', 'profit',
       'un_profitability', 'original_language_english', 'american_film',
       'english_language'],
      dtype='object')
Index(['vote_average', 'vote_count', 'runtime', 'popularity', 'actor_avg',
       'actor_med', 'actor_dev', 'production_avg', 'production_med',
       'production_dev', 'release_year', 'release_month',
       'original_title_matches', 'un_profitability',
       'original_language_english', 'american_film', 'english_language'],
      dtype='object')


In [8]:
y = df["un_profitability"]
X = df.drop(columns=["un_profitability"])

# Creating an 80/20 split 
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.20, random_state=14)
print(f"X_train size: {X_train.shape}")
print(f"X_test size: {X_test.shape}")


X_train size: (13465, 16)
X_test size: (3367, 16)


In [9]:
naive_bayes_model = GaussianNB()
naive_bayes_model.fit(X_train, Y_train)

Y_prediction = naive_bayes_model.predict(X_test)

print("Classification Report:\n", classification_report(Y_test, Y_prediction))

# accuracy = accuracy_score(Y_test, Y_prediction)
# precision = precision_score(Y_test, Y_prediction)
# recall = recall_score(Y_test, Y_prediction)
# f1 = f1_score(Y_test, Y_prediction)

# print(f"Accuracy: {accuracy:.2f}")
# print(f"Precision: {precision:.2f}")
# print(f"Recall: {recall:.2f}")
# print(f"F1 Score: {f1:.2f}")



Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.20      0.32      2144
           1       0.39      0.87      0.54      1223

    accuracy                           0.45      3367
   macro avg       0.56      0.54      0.43      3367
weighted avg       0.61      0.45      0.40      3367



In [10]:
from sklearn.model_selection import StratifiedShuffleSplit  
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)
import numpy as np
k = 10                                 # change to whatever # repeats you want
sss = StratifiedShuffleSplit(
    n_splits=k,                          # k train/test draws
    test_size=0.30,                      # 70 : 30 split each time
    random_state=42                      # reproducible shuffles
)

acc_scores, prec_scores, rec_scores, f1_scores = [], [], [], []

for train_idx, test_idx in sss.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = GaussianNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc_scores.append(accuracy_score(y_test, y_pred))
    prec_scores.append(precision_score(y_test, y_pred, zero_division=0))
    rec_scores.append(recall_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred))

# ---------- 3.  summary  ----------
def show(name, scores):
    print(f"{name:<9}: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

show("Accuracy",  acc_scores)
show("Precision", prec_scores)
show("Recall",    rec_scores)
show("F1-score",  f1_scores)

Accuracy : 0.445 ± 0.019
Precision: 0.373 ± 0.007
Recall   : 0.858 ± 0.029
F1-score : 0.520 ± 0.006
